# EEP 564 Homework 1 Problem 1: DNN and Wine Classification

In problem 1 of our first homework, we will work on the [UCI Wine Dataset](https://uci-ics-mlr-prod.aws.uci.edu/dataset/109/wine), a famous small-scale dataset for early machine learning models. These data are the results of a chemical analysis of wines grown in the same region in Italy but derived from three different cultivars. The analysis determined the quantities of 13 constituents found in each of the three types of wines.

First of all, we start by installing all dependencies required for this problem:

In [ ]:
# If you are running this homework on Google Colab, run the following:
#%pip uninstall -y keras tensorflow tensorflow-model-optimization
%pip install numpy pandas scikit-learn tensorflow<2.21 tensorflow-model-optimization

In [ ]:
# ============================
# Set backend early
# ============================
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

# ============================
# Import libraries
# ============================
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import Input, Model

import tensorflow_model_optimization as tfmot

## Loading and Previewing Wine Dataset

In [ ]:
# Load dataset
column_names = [
    'Class', 'Alcohol', 'Malic acid', 'Ash', 'Alcalinity of ash', 'Magnesium',
    'Total phenols', 'Flavanoids', 'Nonflavanoid phenols', 'Proanthocyanins',
    'Color intensity', 'Hue', 'OD280/OD315 of diluted wines', 'Proline'
]
df = pd.read_csv('wine.data', header=None, names=column_names)

# Number of classes
num_classes = df['Class'].nunique()
print("Number of classes:", num_classes)

# Number of features (excluding the class label)
num_features = df.shape[1] - 1
print("Number of features:", num_features)

# Basic stats: min, max, mean, std
feature_stats = df.describe().T[['min', 'max', 'mean', 'std']]
print("\nFeature statistics:\n", feature_stats)

# Class distribution
class_counts = df['Class'].value_counts().sort_index()
print("\nClass distribution:\n", class_counts)



## Part A: Base Model Training and Evaluation

Now, implement a baseline DNN model for wine classification using the UCI wine dataset. Your code should report training accuracy, test accuracy, and model size of you `float32` model.

In [ ]:
# Step 1: Drop the 'Class' column from the feature set and store it separately
# - Assign features to variable X
# - Subtract 1 from class labels to convert them to 0-based indexing
# - Assign class labels to variable y

# <-- Enter your code here <--#

In [ ]:
# Step 2: Perform a train-test split (70% train, 30% test) using random_state=42

# <-- Enter your code here <--#

In [ ]:
# Step 3: Use StandardScaler to normalize the features
# - Fit on X_train and transform both X_train and X_test

# <-- Enter your code here <--#

In [ ]:
# Step 4: Use one-hot encoding for y_train and y_test
# - Use keras.utils.to_categorical

# <-- Enter your code here <--#

In [ ]:
# Step 5: Define a Sequential model with the following architecture:
# - Dense(64, relu)
# - Dense(32, relu)
# - Dense(3, softmax)  # 3-class classification

# <-- Enter your code here <--#

In [ ]:
# Step 6: Compile using Adam optimizer, categorical_crossentropy loss, and accuracy metric
# - Train for 20 epochs with batch_size=8 and validation_split=0.2

# <-- Enter your code here <--#

In [ ]:
# Step 7: Evaluate the model on test data and print:
# - Accuracy
# - Classification report
# - Confusion matrix

# <-- Enter your code here <--#

In [ ]:
# Step 8: Convert the trained model to TFLite format and save it as "model_base.tflite"
# - Print the file size in kilobytes

# <-- Enter your code here <--#


## Part B: Quantization

Quantize the baseline model from part A using fixed data type quantization with `float16` and `int8`, as well as dynamic range quantization. You should report the same set of metrics as part A.

In [ ]:
def quantize_and_evaluate(model, X_test, y_test_cat, quant_type, filename):
    # Create the TFLite converter from the trained Keras model
    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    # Set supported ops
    converter.target_spec.supported_ops = [
        tf.lite.OpsSet.TFLITE_BUILTINS,
        tf.lite.OpsSet.SELECT_TF_OPS
    ]
    converter._experimental_lower_tensor_list_ops = False

    # Step 1: Apply quantization settings
    if quant_type == 'int8':
        # (a) Enable default optimizations
        # (b) Define a representative dataset generator (e.g., first 100 samples from X_train_scaled)
        # (c) Set inference_input_type and inference_output_type to tf.int8

        # <-- Enter your code here <--#
        pass

    elif quant_type == 'float16':
        # (a) Enable default optimizations
        # (b) Set supported_types to [tf.float16]

        # <-- Enter your code here <--#
        pass

    elif quant_type == 'dynamic':
        # (a) Enable default optimizations

        # <-- Enter your code here <--#
        pass

    # Step 2: Convert the model and save it to the provided filename

    # <-- Enter your code here <--#

    # Step 3: Run Inference
    # Complete the following:
    # - Use tf.lite.Interpreter to load the TFLite model
    # - Allocate tensors
    # - Get input/output tensor details
    # - If input is quantized (dtype=int8), quantize test input accordingly
    # - If output is quantized (dtype=int8), dequantize predictions
    # - Collect predictions into y_pred (use np.argmax to get class index)
    # - Compare with y_true = np.argmax(y_test_cat, axis=1)

    # <-- Enter your code here: implement Step 5 - TFLite inference <--#

    # Step 4: Report results
    print(f"\n📦 {quant_type.upper()} TFLite Model Size: {os.path.getsize(filename) / 1024:.2f} KB")

    # <-- Enter your code here: print classification_report and confusion_matrix <--#


In [ ]:
# Step 5: Use the function above to create and evaluate three quantized models:
# - 'int8' → save as 'model_int8.tflite'
# - 'float16' → save as 'model_float16.tflite'
# - 'dynamic' → save as 'model_dynamic.tflite'

# <-- Enter your code here <--#

## Part C: Pruning

Now apply pruning to **the baseline model from part A** and evaluate the size and performance of the pruned model. We recommend you to perform pruning with `PolynomialDecay` which we have covered in class.

In [ ]:
# Step 1: Define a pruning schedule using tfmot.sparsity.keras.PolynomialDecay
# HINT:
# - Use initial_sparsity = 0.5 and final_sparsity = 0.7
# - Set end_step to total training steps (approx. dataset_size / batch_size * epochs)

# <-- Enter your code here <--#

In [ ]:
# Step 2: Build a Sequential model with 3 pruned Dense layers:
# - Dense(64, relu)
# - Dense(32, relu)
# - Dense(3, softmax)
# Make sure each Dense layer is wrapped with prune_low_magnitude()

# <-- Enter your code here <--#

In [ ]:
# Step 3: Compile the model with categorical_crossentropy and accuracy
# - Train for 10 epochs with batch_size=8 and validation_split=0.2
# - Add tfmot.sparsity.keras.UpdatePruningStep() to the callbacks list

# <-- Enter your code here <--#

In [ ]:
# Step 4: Do any necessary post-processing (if needed) on the pruned model
# and save it using appropriate specifications to a TFLite file named "model_pruned.tflite".
# Print the final file size in KB.

# <-- Enter your code here <--#

In [ ]:
# Step 5: Evaluate using the stripped model
# - Use np.argmax for predictions
# - Print classification_report and confusion_matrix

# <-- Enter your code here <--#

## Part D: Knowledge Distillation

Now apply output-based Knowledge Distillation using **the baseline model from part A** as the teacher and a smaller model as student. Evaluate and report performance and model size of the student.

Again, we recommend you to use the Knowledge Distillation loss we covered in class. If however you decide to use an alternative loss, please indicate that plus why you construct the loss in an alternative way.

In [ ]:
# Step 1: Define a Sequential model for Student with:
# - Dense(32, relu)
# - Dense(16, relu)
# - Dense(3, softmax)

# <-- Enter your code here <--#

In [ ]:
# Step 2: Use model.predict() on X_train_scaled to obtain teacher soft labels

# <-- Enter your code here <--#

In [ ]:
# Step 3:
# (a) Concatenate hard (y_train_cat) and soft (teacher_preds_soft) labels along axis=1
#     to create a combined label for distillation
# (b) Define a custom distillation_loss() function that:
#     - Splits y_true_combined into y_true_hard and y_true_soft
#     - Computes two losses (both using categorical_crossentropy)
#     - Combines them with a weight factor alpha = 0.5

# Hint: Use slicing [:, :3] and [:, 3:] to split the combined labels

# <-- Enter your code here <--#

def distillation_loss(y_true_combined, y_pred):

    # <-- Enter your code here: implement hard/soft label separation and weighted loss <--#
    pass

In [ ]:
# Step 4: Compile the student model with Adam optimizer and distillation_loss
# - Train for 10 epochs, batch_size=8, validation_split=0.2

# <-- Enter your code here <--#

In [ ]:
# Step 5: Convert the student model to TFLite
# - Use appropriate settings for classification models
# - Save as "model_kd.tflite"
# - Print the file size in KB

# <-- Enter your code here <--#

In [ ]:
# Step 7: Use student_model.predict() to obtain predictions on X_test_scaled
# - Print classification_report and confusion_matrix

# <-- Enter your code here <--#

## Part E: Possibility of Further Model Size Reduction

Can you **further reduce the model size** beyond the smallest model obtained in parts **(b)**, **(c)**, or **(d)**, **without sacrificing significant classification performance**?

Your task is to:

1. **Analyze and compare** the results from previous parts: Which model had the smallest size? Which performed best?

2. **Propose a strategy** that combines or enhances techniques learned so far.

3. **Implement** your proposed solution.

4. **Evaluate** the resulting model using both:
   - TFLite model size (in KB)
   - Classification performance (accuracy and report)

5. **Justify your results:**
   - If further size reduction is **not** possible without major loss of accuracy, explain why using evidence from your experiments.
   - If you succeed in reducing the size **further**, highlight what change made the biggest difference.


In [ ]:
# <-- Justify your answer and if needed enter your code here <--#